In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
import random
from matplotlib import colors
import os, glob
import colorsys

In [ ]:
from matplotlib import font_manager
# Manually register the font file with matplotlib's fontManager.
font_manager.fontManager.addfont('../../data/Arial.ttf')

# Verify the font was registered.
print([f.name for f in font_manager.fontManager.ttflist if 'Arial' in f.name])
plt.rcParams['font.family'] = 'Arial'

In [ ]:
def log_fit(x, a, b):
    return a + b * np.log(x)

In [ ]:
files = glob.glob('../../data/regression_outputs/regmodels_spatial_self/Sampling_kcenter/*/*/Fuse/Token_Concat_spatial_self_Spatial/results.csv')
print(len(files))
df_list = []
for file in files:
    tmp_df = pd.read_csv(file)
    country = file.split('/')[-5]
    city = file.split('/')[-4]
    label_sdg_file = f"../../data/processed/0labels/{country}.csv"
    labels_sdg = pd.read_csv(label_sdg_file)
    tmp_df = pd.merge(tmp_df, labels_sdg[['ID', 'SDG']], left_on='target', right_on='ID')
    tmp_df['country'] = country
    tmp_df['city'] = city
    df_list.append(tmp_df)
df = pd.concat(df_list, ignore_index=True)
df

In [ ]:
df.drop(['target', 'k', 'ID', 'country', 'city'], axis=1, inplace=True)
grouped = df.groupby(['ratio', 'SDG']).mean().reset_index()
grouped

In [ ]:
sdg_numbers = grouped['SDG'].unique().tolist()
sdg_numbers

In [ ]:
sdg_colors = {
    "1": "#e5243b",
    "3": "#4C9F38",
    "4": "#C5192D",
    "5": "#FF3A21",
    "6": "#26BDE2",
    "8": "#A21942",
    "9": "#FD6925",
    "10": "#DD1367",
    "11": "#FD9D24",
    "13": "#3F7E44",
    "16": "#00689D"
}

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

# Stores the sampling ratio at which R^2 = 0.8 is reached.
r2_08_points = {}

for sdg in sdg_numbers:
    sdg_df = grouped[grouped['SDG'] == sdg]
    
    x = sdg_df['ratio']
    y = sdg_df['all_r2']
    
    try:
        params_r2, covariance = curve_fit(log_fit, x, y)
        
        # Sampling ratio (in [0, 1]) needed to reach R^2 = 0.8.
        x_08 = np.exp((0.8 - params_r2[0]) / params_r2[1]) 
        
        # Only record x_08 if it falls in a reasonable range (0-100%).
        if 0 < x_08 <= 1.0:
            r2_08_points[sdg] = x_08
        
        print(f'SDG {sdg} reaches R2=0.8 at ratio: {x_08:.4f}')
        
        # Compute parameter standard errors.
        perr_r2 = np.sqrt(np.diag(covariance))
        ci_upper = params_r2 + 1.96 * perr_r2
        ci_lower = params_r2 - 1.96 * perr_r2
        
        # Plot the fitted curve.
        x_fit = np.linspace(0, 1, 1000)
        y_fit_r2 = log_fit(x_fit, *params_r2)
        
        ax.plot(x_fit*100, y_fit_r2, color=sdg_colors[str(sdg)], alpha=1, linewidth=2)
        
        a, b = params_r2
        # print(f'SDG {sdg} equation: R² = {a:.2f} + {b:.2f} ln(ratio)')
        
    except Exception as e:
        print(f"Fit failed for SDG {sdg}: {e}")

# ==========================================
# New: find the SDG with the highest / lowest ratio and annotate it.
# ==========================================
if r2_08_points:
    # SDG with the smallest ratio (fastest to reach 0.8).
    min_sdg = min(r2_08_points, key=r2_08_points.get)
    min_ratio = r2_08_points[min_sdg] * 100 # to percentage
    
    # SDG with the largest ratio (slowest to reach 0.8).
    max_sdg = max(r2_08_points, key=r2_08_points.get)
    max_ratio = r2_08_points[max_sdg] * 100 # to percentage

    # 1. Vertical dashed line at the minimum ratio.
    ax.vlines(x=min_ratio, ymin=0, ymax=0.8, color=sdg_colors[str(min_sdg)], 
              linestyle='--', linewidth=1.5, alpha=1)
    # Label numbers with a small offset to avoid overlap.
    ax.text(min_ratio, 0.05, f'{min_ratio:.2f}%', color=sdg_colors[str(min_sdg)], 
            fontsize=12, ha='right', va='bottom', fontweight='bold', rotation=90)

    # 2. Vertical dashed line at the maximum ratio.
    ax.vlines(x=max_ratio, ymin=0, ymax=0.8, color=sdg_colors[str(max_sdg)], 
              linestyle='--', linewidth=1.5, alpha=1)
    # Annotate the number.
    ax.text(max_ratio, 0.05, f'{max_ratio:.2f}%', color=sdg_colors[str(max_sdg)], 
            fontsize=12, ha='left', va='bottom', fontweight='bold', rotation=90)

# ==========================================
# Style.
# ==========================================
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.spines['bottom'].set_linewidth(2)
ax.spines['left'].set_linewidth(2)

ax.set_xlim(0, 100)
ax.set_ylim(0, 1)

ax.tick_params(axis='both', which='major', labelsize=16)
# Reference line at 0.8.
ax.axhline(y=0.8, color='gray', linestyle='--', linewidth=1.5, alpha=0.5)

ax.set_xlabel('Average surveyed areas (%)', fontsize=16)
ax.set_ylabel('Overall $R^2$', fontsize=16)

plt.tight_layout()
plt.savefig('../../data/figure_assets/fig1_sdg_curve.svg', format='svg', bbox_inches='tight')
plt.show()